# 365 Probabilidades - Dia #142
## Qual a probabilidade de o seu sobrenome desaparecer?

**Tipo:** Estrutural
**Data de publicação:** 2026-11-02
**Ferramenta:** Python
**Decisão analisada:** O que a gente deixa com o nome?
**Hashtag:** #365Probabilidades #Dia142

---

### 📖 A História

Dia de Finados é dia de ler nomes. Nas lápides, nos ossários, nas placas de rua, os
sobrenomes ficam depois das pessoas. Alguns parecem eternos. Outros ninguém mais ouve.

Em 1873, Francis Galton publicou uma pergunta num jornal de matemática. As famílias
inglesas de nome ilustre estavam sumindo, e ele queria saber se isso era azar ou
aritmética. Um reverendo chamado Henry William Watson resolveu a conta e os dois
publicaram juntos, em 1875. Watson concluiu que toda família acaba. Estava errado, e o
erro levou cinquenta anos para ser corrigido. Mas a conta dele, com o conserto, é até hoje
a forma de responder quanto tempo um nome dura.

---

### 📚 O Conceito: o processo de ramificação

Um sobrenome passa de uma geração para a outra como uma árvore cresce. Cada homem tem um
número de filhos homens, sorteado de uma distribuição. Cada filho repete o sorteio. Se em
alguma geração ninguém tem filho homem, a linhagem acaba.

A matemática disso é o **processo de Galton e Watson**. O que decide o destino é um número
só: a média de filhos homens por homem, que a gente chama de **m**. Se m é maior que 1, a
linhagem tem uma chance de sobreviver para sempre. Se m é menor ou igual a 1, a extinção é
certa; a única pergunta é quando. Watson errou porque a conta dele dava extinção certa em
qualquer caso. Steffensen mostrou em 1930 que a solução dele só valia para m até 1.

No Brasil de 1960, com 6,28 filhos por mulher, m passava de 3. No Brasil do Censo 2022,
com 1,55 filho por mulher e metade dos nascidos meninos, m é 0,79. Cruzamos a linha há
cerca de uma geração.

**O que isso prevê:** com todas as linhagens condenadas, os nomes raros somem primeiro.
Um sobrenome carregado por mil homens é, na prática, mil linhagens independentes, e só
acaba quando a última delas acaba. Por isso o que sobra num país de fecundidade baixa são
os nomes que já eram grandes. A Coreia, onde três sobrenomes respondem por 45% do país,
chegou lá por história (famílias comuns adotaram nomes de elite ao longo de séculos), não
só por sorteio; o modelo serve para mostrar a assimetria entre o nome raro e o nome
comum, não para explicar a Coreia.

---

### 🧮 O Modelo

Cinco camadas.

1. **Uma linhagem, hoje.** A chance de o sobrenome de um homem acabar em 1, 2, 3, 5 e 10
   gerações, com a fecundidade de 2022, e quantas gerações a linhagem dura em média.
2. **A mesma linhagem em 1960.** A conta com 6,28 filhos por mulher, para ver o que mudou.
3. **Por grupo.** As fecundidades do Censo 2022 por cor ou raça, de 1,22 a 2,48.
4. **Se o nome passar por qualquer filho.** A versão em que filhas também levam o nome.
5. **Nome raro e nome comum.** Quanto tempo dura um sobrenome carregado por 1, 10, 100,
   mil e 10 mil homens, e por que os raros somem primeiro.

**Tabela de fontes**

| Fonte | O que é | Número usado |
|---|---|---|
| Galton, F. & Watson, H. W. (1875). On the probability of the extinction of families. *Journal of the Anthropological Institute*, 4, 138-144 | o modelo original e a pergunta de 1873 | a estrutura da conta; a correção de Steffensen (1930) |
| IBGE, Censo Demográfico 2022, Fecundidade e migração, resultados preliminares da amostra (junho de 2025) | registro censitário, Brasil | TFT 1,55 (1960: 6,28); 16,2% das mulheres sem filhos; brancas 1,35, pretas 1,59, pardas 1,68, amarelas 1,22, indígenas 2,48 |
| SINASC 2019-2024 (contagem feita no #136) | registro de nascimentos | razão de sexo ao nascer, 51,2% de meninos |
| Statistics Korea, Censo 2015 | registro censitário, Coreia do Sul | Kim 21,5%, Lee 14,7%, Park 8,4%; os três, 44,6% |
| Yoshida, H. (Universidade de Tohoku, 2024), projeção divulgada na imprensa | projeção, Japão | com a lei do sobrenome único e a tendência atual, em 2531 todos os japoneses se chamariam Sato; citada como ilustração, não como dado |

Leitura: Galton e Watson lidos na reimpressão; IBGE e Statistics Korea lidos nos releases
oficiais; a projeção japonesa só na cobertura de imprensa (declarado).

**Nota metodológica.** ×0,80 não se aplica: fecundidade e sexo ao nascer são registro,
não opinião. O número de filhos por homem é sorteado de uma Poisson com a média do Censo,
que é a hipótese clássica do modelo e dá 21% de homens sem filhos, perto dos 16% de
mulheres sem filhos que o Censo mede sem distinguir idade. A sensibilidade a uma
distribuição mais espalhada (binomial negativa) está na camada 1. Cordas: #136 (a razão
de sexo veio de lá) e #147 (o paradoxo do aniversário, outro dia de conta pura).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

SEMENTE = 42
rng = np.random.default_rng(SEMENTE)

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'

def mil(n):
    return f"{int(round(n)):,}".replace(",", ".")

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

def pct(x, casas=0):
    return vir(100 * x, casas) + "%"

print("✅ Bibliotecas carregadas · semente", SEMENTE)

✅ Bibliotecas carregadas · semente 42


In [2]:
# --- DADOS DA LITERATURA ---

# IBGE, Censo 2022 (fecundidade e migração, resultados preliminares da amostra, junho de 2025)
TFT_2022 = 1.55
TFT_1960 = 6.28
P_SEM_FILHOS_2022 = 0.162          # mulheres sem filhos, 2022 (10,0% em 2000)
TFT_GRUPO = {"amarelas": 1.22, "brancas": 1.35, "pretas": 1.59, "pardas": 1.68, "indígenas": 2.48}

# SINASC 2019-2024 (#136): razão de sexo ao nascer
P_MENINO = 0.512

# Coreia do Sul, Censo 2015 (Statistics Korea)
COREIA = {"Kim": 0.215, "Lee": 0.147, "Park": 0.084}

# Galton & Watson 1875: o modelo; Steffensen 1930: a correção (extinção certa só se m <= 1)
ANOS_POR_GERACAO = 28.1            # idade média da mãe ao ter filho, Censo 2022 (26,3 em 2000)

print("=" * 70)
print("  DADOS · SOBRENOMES, FECUNDIDADE E RAMIFICAÇÃO")
print("=" * 70)
print(f"\n  Filhos por mulher (IBGE): {vir(TFT_1960, 2)} em 1960 → {vir(TFT_2022, 2)} em 2022")
print(f"  Mulheres sem filhos em 2022: {pct(P_SEM_FILHOS_2022, 1)}")
print("  Por cor ou raça: " + "; ".join(f"{k} {vir(v, 2)}" for k, v in TFT_GRUPO.items()))
print(f"  Meninos entre os nascidos (SINASC): {pct(P_MENINO, 1)}")
print(f"  Coreia 2015: " + "; ".join(f"{k} {pct(v, 1)}" for k, v in COREIA.items()) + f"; os três, {pct(sum(COREIA.values()), 1)}")
print(f"  Uma geração, aqui: {vir(ANOS_POR_GERACAO)} anos (idade média da mãe ao ter filho)")
print("=" * 70)

  DADOS · SOBRENOMES, FECUNDIDADE E RAMIFICAÇÃO

  Filhos por mulher (IBGE): 6,28 em 1960 → 1,55 em 2022
  Mulheres sem filhos em 2022: 16,2%
  Por cor ou raça: amarelas 1,22; brancas 1,35; pretas 1,59; pardas 1,68; indígenas 2,48
  Meninos entre os nascidos (SINASC): 51,2%
  Coreia 2015: Kim 21,5%; Lee 14,7%; Park 8,4%; os três, 44,6%
  Uma geração, aqui: 28,1 anos (idade média da mãe ao ter filho)


In [3]:
# --- O MODELO ---

# A função geradora da Poisson: G(s) = exp(m (s - 1)). A chance de extinção até a geração n
# é q_n = G(q_{n-1}), com q_0 = 0. A extinção eventual é o menor ponto fixo q = G(q).
def extincao_por_geracao(m, n_ger=30):
    q = [0.0]
    for _ in range(n_ger):
        q.append(np.exp(m * (q[-1] - 1)))
    return np.array(q)

def extincao_eventual(m):
    if m <= 1:
        return 1.0
    return optimize.brentq(lambda q: np.exp(m * (q - 1)) - q, 1e-12, 1 - 1e-12)

def vida_media_geracoes(m, n_ger=400):
    # soma das probabilidades de ainda existir em cada geração (só converge para m < 1)
    q = extincao_por_geracao(m, n_ger)
    return (1 - q).sum()

GER = [1, 2, 3, 5, 10, 20]

# CAMADA 1 · uma linhagem hoje: só filhos homens levam o nome
m_2022 = TFT_2022 * P_MENINO
q_2022 = extincao_por_geracao(m_2022)
vida_2022 = vida_media_geracoes(m_2022)
print("=" * 70)
print("  CAMADA 1 · UMA LINHAGEM HOJE (nome passa pelos filhos homens)")
print("=" * 70)
print(f"\n  filhos homens por homem, em média: m = {vir(TFT_2022, 2)} × {vir(P_MENINO, 3)} = {vir(m_2022, 2)}  (abaixo de 1: extinção certa)")
for g in GER:
    print(f"  acaba em até {g:>2d} geração(ões): {pct(q_2022[g])}")
print(f"  dura em média {vir(vida_2022)} gerações, cerca de {mil(vida_2022 * ANOS_POR_GERACAO)} anos")
# sensibilidade: binomial negativa com a mesma média e mais espalhamento (variância = média × 1,5)
def extincao_nb(m, disp=1.5, n_ger=30):
    # NB com média m e variância disp*m: G(s) = (1 + (disp-1)(1-s))^(-m/(disp-1))
    q = [0.0]
    for _ in range(n_ger):
        q.append((1 + (disp - 1) * (1 - q[-1])) ** (-m / (disp - 1)))
    return np.array(q)
q_nb = extincao_nb(m_2022)
print(f"  com mais espalhamento (binomial negativa): acaba em 1 geração {pct(q_nb[1])}, em 5 {pct(q_nb[5])}, em 10 {pct(q_nb[10])}")
print(f"  homens sem filho nenhum no modelo: Poisson {pct(np.exp(-TFT_2022))} · Censo (mulheres sem filhos) {pct(P_SEM_FILHOS_2022, 1)}")

# CAMADA 2 · a mesma linhagem em 1960
m_1960 = TFT_1960 * P_MENINO
q_1960 = extincao_por_geracao(m_1960)
qe_1960 = extincao_eventual(m_1960)
print("\n" + "=" * 70)
print("  CAMADA 2 · A MESMA LINHAGEM EM 1960")
print("=" * 70)
print(f"\n  m = {vir(TFT_1960, 2)} × {vir(P_MENINO, 3)} = {vir(m_1960, 2)}  (acima de 1: pode durar para sempre)")
for g in GER:
    print(f"  acaba em até {g:>2d} geração(ões): {pct(q_1960[g], 1)}")
print(f"  chance de acabar algum dia: {pct(qe_1960, 1)}; chance de durar para sempre: {pct(1 - qe_1960, 1)}")
# em que fecundidade a linha cruza m = 1?
TFT_LIMITE = 1 / P_MENINO
print(f"  a linha é cruzada com {vir(TFT_LIMITE, 2)} filhos por mulher; o Brasil passou por ela por volta de 2010 (TFT 2,38 em 2000, 1,90 em 2010)")

# CAMADA 3 · por grupo de cor ou raça
print("\n" + "=" * 70)
print("  CAMADA 3 · POR GRUPO (Censo 2022)")
print("=" * 70)
grupos = {}
for k, t in TFT_GRUPO.items():
    m = t * P_MENINO
    q = extincao_por_geracao(m)
    qe = extincao_eventual(m)
    grupos[k] = (t, m, q, qe)
    print(f"  {k:<10s} TFT {vir(t, 2)}  m {vir(m, 2)}  acaba em 5 gerações {pct(q[5])}  em 10 {pct(q[10])}  algum dia {pct(qe)}")

# CAMADA 4 · se qualquer filho levar o nome
m_qualquer = TFT_2022
q_qualquer = extincao_por_geracao(m_qualquer)
qe_qualquer = extincao_eventual(m_qualquer)
print("\n" + "=" * 70)
print("  CAMADA 4 · SE O NOME PASSAR POR QUALQUER FILHO")
print("=" * 70)
print(f"\n  m = {vir(m_qualquer, 2)} (acima de 1)")
for g in GER:
    print(f"  acaba em até {g:>2d} geração(ões): {pct(q_qualquer[g])}")
print(f"  chance de acabar algum dia: {pct(qe_qualquer)}; de durar para sempre: {pct(1 - qe_qualquer)}")

# CAMADA 5 · nome raro e nome comum: um sobrenome com N homens só acaba quando as N linhagens acabam
TAMANHOS = [1, 10, 100, 1_000, 10_000]
q_long = extincao_por_geracao(m_2022, 400)
def ger_para_metade(N):
    # primeira geração em que a chance de o nome inteiro ter acabado passa de 50%
    p = q_long ** N
    idx = np.argmax(p >= 0.5)
    return int(idx) if p[idx] >= 0.5 else None
print("\n" + "=" * 70)
print("  CAMADA 5 · NOME RARO E NOME COMUM (fecundidade de 2022)")
print("=" * 70)
nome_tab = {}
for N in TAMANHOS:
    p = q_long ** N
    g50 = ger_para_metade(N)
    nome_tab[N] = (p, g50)
    print(f"  {mil(N):>6s} homens hoje: acabou em 5 gerações {pct(p[5])} · em 10 {pct(p[10])} · em 20 {pct(p[20])} · passa de 50% na geração {g50} (~{mil(g50 * ANOS_POR_GERACAO)} anos)")
print("  os raros somem primeiro; os grandes ficam, e é assim que um país de fecundidade baixa perde variedade de nomes")
print("=" * 70)

  CAMADA 1 · UMA LINHAGEM HOJE (nome passa pelos filhos homens)

  filhos homens por homem, em média: m = 1,55 × 0,512 = 0,79  (abaixo de 1: extinção certa)
  acaba em até  1 geração(ões): 45%
  acaba em até  2 geração(ões): 65%
  acaba em até  3 geração(ões): 76%
  acaba em até  5 geração(ões): 87%
  acaba em até 10 geração(ões): 97%
  acaba em até 20 geração(ões): 100%
  dura em média 2,9 gerações, cerca de 81 anos
  com mais espalhamento (binomial negativa): acaba em 1 geração 53%, em 5 90%, em 10 97%
  homens sem filho nenhum no modelo: Poisson 21% · Censo (mulheres sem filhos) 16,2%

  CAMADA 2 · A MESMA LINHAGEM EM 1960

  m = 6,28 × 0,512 = 3,22  (acima de 1: pode durar para sempre)
  acaba em até  1 geração(ões): 4,0%
  acaba em até  2 geração(ões): 4,6%
  acaba em até  3 geração(ões): 4,6%
  acaba em até  5 geração(ões): 4,7%
  acaba em até 10 geração(ões): 4,7%
  acaba em até 20 geração(ões): 4,7%
  chance de acabar algum dia: 4,7%; chance de durar para sempre: 95,3%
  a linh

In [4]:
# --- VISUALIZAÇÃO ---
RODAPE = 'Fontes: Galton & Watson (1875); IBGE, Censo 2022; SINASC · modelo próprio, semente 42 | #365Probabilidades'
gs = np.arange(0, 21)

# GRÁFICO 1 · a chance de acabar, geração a geração, 2022 contra 1960
fig1, ax1 = plt.subplots(figsize=(12, 7.5))
ax1.plot(gs, q_2022[:21] * 100, color=VERMELHO, lw=3, marker='o', label=f'Brasil 2022 · 1,55 filho por mulher · m = {vir(m_2022, 2)}')
ax1.plot(gs, q_qualquer[:21] * 100, color=DOURADO, lw=2.5, ls='--', marker='s', markersize=5, label=f'2022, se qualquer filho levar o nome · m = {vir(m_qualquer, 2)}')
ax1.plot(gs, q_1960[:21] * 100, color=VERDE, lw=3, marker='o', label=f'Brasil 1960 · 6,28 filhos por mulher · m = {vir(m_1960, 2)}')
ax1.axhline(100, color='#333', lw=1, ls=':')
for g in (1, 5, 10):
    ax1.annotate(pct(q_2022[g]), (g, q_2022[g] * 100), xytext=(0, 10), textcoords='offset points', ha='center', fontsize=12, fontweight='bold', color=VERMELHO)
ax1.annotate(pct(qe_1960, 1) + ' algum dia', (20, q_1960[20] * 100), xytext=(0, 10), textcoords='offset points', ha='right', fontsize=11, color=VERDE)
ax1.set_xticks(gs[::2]); ax1.set_xlim(0, 20.5); ax1.set_ylim(0, 105)
ax1.set_xlabel(f'gerações (uma geração ≈ {vir(ANOS_POR_GERACAO, 0)} anos)', fontsize=12); ax1.set_ylabel('chance de a linhagem já ter acabado (%)', fontsize=12)
ax1.legend(fontsize=11, loc='center right')
ax1.set_title(f'Com a fecundidade de hoje, o sobrenome de um homem tem {pct(q_2022[5])} de chance de acabar em cinco gerações\n'
              f'em 1960 eram {pct(q_1960[5], 1)}; a linha que separa os dois mundos é um filho homem por homem', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-142-grafico-01-geracao-a-geracao.png', dpi=150, bbox_inches='tight'); plt.close()
print("✅ Gráfico 1 salvo!")

# GRÁFICO 2 · a linha do m = 1: chance de durar para sempre em função da fecundidade
fig2, ax2 = plt.subplots(figsize=(12, 7))
tfts = np.linspace(1.0, 7.0, 300)
sobrevive = np.array([1 - extincao_eventual(t * P_MENINO) for t in tfts])
ax2.plot(tfts, sobrevive * 100, color='#222222', lw=3)
ax2.axvline(TFT_LIMITE, color=VERMELHO, lw=2, ls='--'); ax2.text(TFT_LIMITE + 0.05, 60, f'{vir(TFT_LIMITE, 2)} filhos por mulher\num filho homem por homem', fontsize=11, color=VERMELHO)
desl = {'amarelas': 42, 'brancas': 30, 'pretas': 18, 'pardas': 6, 'indígenas': 8}
for k, (t, m, q, qe) in grupos.items():
    ax2.scatter([t], [(1 - qe) * 100], s=90, color=DOURADO, zorder=5)
    ax2.annotate(f"{k} ({vir(t, 2)})", (t, (1 - qe) * 100), xytext=(0, desl[k]), textcoords='offset points', ha='center', fontsize=10.5, color='#6b5a1e',
                 arrowprops=dict(arrowstyle='-', color='#c9b27a', lw=0.8) if k != 'indígenas' else None)
ax2.scatter([TFT_2022], [0], s=160, color=VERMELHO, zorder=6); ax2.annotate('Brasil 2022', (TFT_2022, 0), xytext=(0, -22), textcoords='offset points', ha='center', fontsize=12, fontweight='bold', color=VERMELHO)
ax2.scatter([TFT_1960], [(1 - qe_1960) * 100], s=160, color=VERDE, zorder=6); ax2.annotate('Brasil 1960', (TFT_1960, (1 - qe_1960) * 100), xytext=(-10, -18), textcoords='offset points', ha='right', fontsize=12, fontweight='bold', color=VERDE)
ax2.set_xlabel('filhos por mulher', fontsize=12); ax2.set_ylabel('chance de a linhagem durar para sempre (%)', fontsize=12)
ax2.set_xlim(1, 7); ax2.set_ylim(-8, 100)
ax2.set_title('A linha de Galton e Watson: abaixo de um filho homem por homem, a extinção é certa\nem dourado, os grupos do Censo 2022; só os indígenas ainda estão do lado de cima', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-142-grafico-02-a-linha.png', dpi=150, bbox_inches='tight'); plt.close()
print("✅ Gráfico 2 salvo!")

# GRÁFICO 3 · cem linhagens sorteadas, dez gerações
fig3, ax3 = plt.subplots(figsize=(12, 7))
N_LIN, G3 = 100, 10
tam = np.ones((N_LIN, G3 + 1), dtype=int)
for g in range(1, G3 + 1):
    tam[:, g] = np.array([rng.poisson(m_2022, size=n).sum() if n > 0 else 0 for n in tam[:, g - 1]])
vivas = (tam > 0).sum(axis=0)
ax3.bar(range(G3 + 1), vivas, color=[VERMELHO] + [VERDE] * G3, alpha=0.9, width=0.65)
ax3.plot(range(G3 + 1), 100 * (1 - q_2022[:G3 + 1]), color='#222222', lw=2, ls='--', marker='o', markersize=5, label='o que a fórmula de Galton e Watson prevê')
for g, v in enumerate(vivas):
    ax3.text(g, v + 1.5, str(int(v)), ha='center', fontsize=12, fontweight='bold')
ax3.set_xticks(range(G3 + 1)); ax3.set_xlabel('gerações', fontsize=12); ax3.set_ylabel('sobrenomes ainda vivos, de 100', fontsize=12)
ax3.set_ylim(0, 108); ax3.legend(fontsize=11, loc='upper right')
ax3.set_title(f'Cem sobrenomes que começam com um homem hoje, sorteados geração a geração\n'
              f'{int(vivas[1])} passam da primeira geração, {int(vivas[5])} da quinta, {int(vivas[G3])} chegam à décima', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-142-grafico-03-cem-linhagens.png', dpi=150, bbox_inches='tight'); plt.close()
print("✅ Gráfico 3 salvo!")

# GRÁFICO 4 · nome raro e nome comum
fig4, ax4 = plt.subplots(figsize=(12, 7))
cores4 = [VERMELHO, '#d9823b', DOURADO, VERDE, '#222222']
gs4 = np.arange(0, 61)
for (N, (p, g50)), c in zip(nome_tab.items(), cores4):
    ax4.plot(gs4, p[:61] * 100, color=c, lw=2.8, label=f'{mil(N)} {"homem" if N == 1 else "homens"} hoje · passa de 50% na geração {g50}')
ax4.axhline(50, color='#333', lw=1, ls=':')
ax4.set_xlabel(f'gerações (uma geração ≈ {vir(ANOS_POR_GERACAO, 0)} anos)', fontsize=12); ax4.set_ylabel('chance de o sobrenome inteiro já ter acabado (%)', fontsize=12)
ax4.set_xlim(0, 60); ax4.set_ylim(0, 102); ax4.legend(fontsize=11, loc='lower right')
ax4.set_title('Os nomes raros somem primeiro: um sobrenome de um homem só tem 45% de chance de acabar em uma geração,\n'
              f'um de 10 mil homens leva {nome_tab[10_000][1]} gerações para chegar aos mesmos 50%', fontsize=13.5, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-142-grafico-04-raro-e-comum.png', dpi=150, bbox_inches='tight'); plt.close()
print("✅ Gráfico 4 salvo!")

✅ Gráfico 1 salvo!
✅ Gráfico 2 salvo!
✅ Gráfico 3 salvo!
✅ Gráfico 4 salvo!


### 💡 O Insight

**Com a fecundidade do Brasil de hoje, o sobrenome de um homem tem 45% de chance de acabar
já na próxima geração, 87% em cinco gerações e 97% em dez. Em 1960 a chance de acabar
algum dia era de 5%.**

O que mudou não foi a sorte, foi a média. Quando cada homem tem em média menos de um
filho homem, e hoje tem 0,79, a extinção de cada linhagem deixa de ser possível e passa a
ser certa. A única pergunta é quando. A conta é de 1875 e continua valendo.

O que isso não quer dizer é que o Brasil vai ficar sem sobrenomes. Um nome carregado por
dez mil homens é dez mil linhagens, e só acaba quando a última acaba. Os raros somem
primeiro, os comuns ficam. É assim que um país de fecundidade baixa perde variedade de
nomes sem perder os nomes.

No Dia de Finados, os nomes nas lápides parecem a parte que fica. A matemática diz que a
maioria deles é a parte que passa. O que fica é outra coisa.

*Quantas gerações para trás você consegue dizer o sobrenome inteiro? E para a frente?*

---

### ⚠️ Limitações

**Do modelo**
- O número de filhos por homem segue uma Poisson com a média do Censo. Famílias reais são
  mais desiguais do que isso, e a binomial negativa da camada 1 mostra que a extinção na
  primeira geração sobe e a de longo prazo quase não muda.
- A média de filhos por homem foi tomada igual à de filhos por mulher. Homens têm filhos
  com mais de uma mulher e vice-versa; a aproximação é razoável para a média, não para a
  distribuição.
- A fecundidade foi mantida constante ao longo das gerações. Ela caiu de 6,28 para 1,55 em
  sessenta anos e pode mudar de novo; o modelo não prevê o futuro, mostra o que a taxa de
  hoje implica.
- "Sobrenome passa pelo filho homem" é a regra do modelo clássico. No Brasil muita gente
  carrega os dois sobrenomes, e a camada 4 mostra a versão em que qualquer filho leva o
  nome.
- A camada do nome comum trata as linhagens do mesmo sobrenome como independentes, o que
  elas são no modelo e quase são na vida real.
- A concentração de sobrenomes na Coreia tem causas históricas (adoção de nomes de elite
  por famílias comuns ao longo da dinastia Joseon); o modelo não a explica e o texto não
  diz que explica.

**Das fontes**
- Os 16,2% de mulheres sem filhos do Censo incluem mulheres jovens que ainda podem ter;
  não é fecundidade completa.
- A projeção japonesa de 2531 é exercício de um pesquisador divulgado pela imprensa, não
  artigo revisado; entra como ilustração.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*